# Versuch 155, step 1: pipeline (point clouds and rasters → tables)

Everything that needs the point clouds or the rasters happens here. The notebook writes **one table per topic with one row per field tree** (or per LiDAR instance) to `outputs/site155_all/tables/`. It contains no statistics and no result figures: those are in `02_results_155.ipynb`, which reads only these tables.

**Ground truth**
- Parzellen 1–26: `155_EBEe_mitVertexund Maßband.xlsx`; Parzellen 27–45: `155_EBEe-P27-45_done_ausg 1.xlsx`
- Trees with an entry in `removal` (taken out in an earlier thinning) are not counted.
- In Parzellen 41–45 the local y of the sheet runs from the opposite plot edge; it is mirrored here (checked in section 3).

**LiDAR**: SegmentAnyTree instances of a before-harvest flight (`BEFORE_FLIGHT`, default 25 Feb 2026). The after-harvest flight (24 Mar 2026) gives the ground model and the harvest evidence.

**Matching**: one global one-to-one assignment of field stems to instance *bases* (stem slice at 1–3 m), with a cost penalty when the harvest evidence disagrees. Field heights are not used for the assignment.

**Classes per field tree**
- `detected`: an instance base within 1.5 m of the stem, and the harvest evidence agrees
- `uncertain`: an instance was assigned, but it is 1.5–3 m away or the harvest evidence disagrees
- `not detected`: no free instance base within 3 m

**Running it**
- Every slow step is cached in its table. A step runs only when its table is missing or `FORCE_RECOMPUTE = True`. With all caches present the notebook takes about a minute.
- The point clouds (300 M points each, about 5 minutes and 20 GB) are loaded once, by the first step that needs them.
- Settings and paths shared with the results notebook are in `modules/notebook_common.py`.

| Section | Writes |
|---|---|
| 2 Instances | `lidar_instances.gpkg`, `point_density_per_parzelle.csv`, DTM and CHM rasters |
| 3 Orientation | `orientation_check.csv` |
| 4 Matching | `field_trees_detection.csv`, `match_distance_only.csv`, `chance_control_matches.csv`, `detection_per_parzelle.csv`, `detection_all_trees.gpkg` |
| 5 Stand structure | `structure_per_tree.csv`, `canopy_per_parzelle.csv` |
| 6 Stems | `stem_form_per_tree.csv`, `column_structure_per_tree.csv`, `tree_profile_all.csv`, `missed_trees_profile.csv` |
| 7 Causes | `match_causes.csv` |
| 8 Heights and DBH | `height_dbh_all_trees.csv`, `photo_ground_offset.csv`, `photo_tree_tops.gpkg` |
| 9 Tiers | `match_tiers_auto.csv`, `match_tiers.csv` |
| 10 QA | figures in `figures/qa_*`, optional clips in `inspect/` |
| 11 Matching with DBH and height | `matching_comparison/*.csv` |
| 12 Stem-seeded refinement | `refined_trees.csv`, `refined_trees/*.laz` |
| 13 Instances without a field tree | `lidar_instances_without_field_tree.csv`, figures in `figures/qa_instances_without_field_tree/` |
| 14 Baseline: local maxima on the CHM | `chm_baseline_tops.csv`, `chm_baseline_matches.csv` |
| 15 18 Mar flight (double grid) | `flight_20260318/`: removed share per instance on 18 and 24 Mar, breast-height slices, DBH per flight |
| 15b SegmentAnyTree in the March flights | `flight_20260318/`: instances per March flight, matching and chance control per flight |

In [ ]:
import json
import sys
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from scipy.spatial import cKDTree
from shapely.geometry import Point

# Project root = the folder that contains 'modules/'
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'modules').is_dir())))

from modules import chm_baseline
from modules import field_tree_matching as ftm
from modules import match_causes
from modules import results_stats as rs
from modules import stem_seeded_refinement as ssr
from modules.coord_transform import fit_parzelle_transforms, parzelle_polygons, transform_trees_local_to_utm
from modules.field_data_excel import load_field_trees_inventory
from modules.lidar_dtm import create_lidar_dtm
from modules.notebook_common import *      # paths, thresholds and plot style, shared with 02_results_155.ipynb
from modules.raster_processing import create_ndsm
from modules.subset import subset_raster
from modules.tree_detection import detect_ndsm_tree_tops
from modules.tree_extraction import extract_ndsm_tree_heights

# --- What to run ---
BEFORE_FLIGHT = '20260225'          # '20260225', or '20260318' (double grid, partly after the harvest)
PARZELLEN = None                    # None = all 45 Parzellen; a list (e.g. [1, 2, 3, 4, 6, 8, 9]) restricts the run
FORCE_RECOMPUTE = False             # True: redo every cached step
QA_CLASSES = ['not detected', 'uncertain']     # QA figure for every field tree of these classes (section 10)
WRITE_INSPECTION_CLIPS = False      # point cloud clips for CloudCompare (section 10); needs about 40 GB of memory
INSPECT_EXTRA = ['4_1040']          # further trees to clip, besides the sample trees that need a look
RUN_MATCHING_COMPARISON = True      # section 11
REFINE_TREES = 'auto'               # section 12: 'auto' (every sample tree and the standing-height trees with an error
                                    # above REFINE_OUTLIER_M), a list such as ['2_1027'], or [] for none; needs a matched instance
TOP_EXTENSION_TREES = ['2_1027']    # section 12: trees whose top is found by closing the gap to the crown base above
                                    # (free refinement + extend_top_into_canopy); their point set and figures are this result
N_UNMATCHED_FIGURES = 10            # section 13: random LiDAR instances without a field tree that get a QA figure

SAT_BEFORE, INSTANCE_DIM, _ = BEFORE_FLIGHTS[BEFORE_FLIGHT]
OUT = output_dir(BEFORE_FLIGHT)
TABLES = OUT / 'tables'
for sub in ['subset', 'rasters', 'tables', 'tables/matching_comparison', 'figures', 'manual_trees', 'inspect', 'refined_trees']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print(f"Before-harvest flight: {BEFORE_FLIGHT} ({SAT_BEFORE.name})")
print(f"Outputs: {OUT}")

## 1. Field inventory

Local plot coordinates are converted to UTM with a per-Parzelle affine fit to the surveyed corner points. The Excel files are read here, before any GeoPackage or raster is opened (reading Excel afterwards crashes in this environment).

In [ ]:
corners_df = pd.read_excel(CORNERS_PATH)
field = load_field_trees_inventory(EXCEL_PATHS)
if PARZELLEN is not None:
    corners_df = corners_df[corners_df['Parzelle'].isin(PARZELLEN)].copy()
    field = field[field['parzelle'].astype(int).isin(PARZELLEN)].reset_index(drop=True)
mirrored_parzellen = [p for p in Y_MIRRORED_PARZELLEN if PARZELLEN is None or p in PARZELLEN]

field['local_y_sheet'] = field['local_y']
plot_width = corners_df.groupby('Parzelle')['x'].max()      # 48 m
plot_depth = corners_df.groupby('Parzelle')['y'].max()      # 30 m
mirrored = field['parzelle'].astype(int).isin(mirrored_parzellen)
field.loc[mirrored, 'local_y'] = field.loc[mirrored, 'parzelle'].astype(int).map(plot_depth) - field.loc[mirrored, 'local_y_sheet']
field['y_mirrored'] = mirrored
field = transform_trees_local_to_utm(field, corners_df)
field['block'] = np.where(field['parzelle'].astype(int) <= 26, 'Parzellen 1-26', 'Parzellen 27-45')
assert field[['utm_x', 'utm_y']].notna().all().all(), "Some trees have no UTM position"

dbh_ok = field['dbh_mm'].between(*DBH_PLAUSIBLE_MM)
field['dbh_class'] = pd.cut(field['dbh_mm'].where(dbh_ok), [0, 200, 300, 400, 500, 2000],
                            labels=['< 200 mm', '200-300 mm', '300-400 mm', '400-500 mm', '> 500 mm'])
field['dbh_class'] = field['dbh_class'].cat.add_categories('no / implausible DBH').fillna('no / implausible DBH')

plots = parzelle_polygons(corners_df, crs=CRS)
corner_rmse = pd.Series({k: v['rmse'] for k, v in fit_parzelle_transforms(corners_df).items()}, name='corner_rmse_m')
aoi = (float(corners_df['UTM_x'].min() - AOI_BUFFER_M), float(corners_df['UTM_y'].min() - AOI_BUFFER_M),
       float(corners_df['UTM_x'].max() + AOI_BUFFER_M), float(corners_df['UTM_y'].max() + AOI_BUFFER_M))

print(f"Trees standing before the 2026 harvest: {len(field)} in {field['parzelle'].nunique()} Parzellen "
      f"({int(field['has_height'].sum())} sample trees with a felled length)")
print(f"Local y mirrored for Parzellen {mirrored_parzellen}: {int(mirrored.sum())} trees")
print(f"AOI: {tuple(round(v, 1) for v in aoi)}  ({aoi[2] - aoi[0]:.0f} x {aoi[3] - aoi[1]:.0f} m)")

## 2. LiDAR tree instances (slow the first time, then cached)

Clips both flights to the AOI, builds the ground model from the after-harvest cloud, flags the before-harvest points that are gone afterwards, and summarizes every SegmentAnyTree instance (base, top, height, share of removed points). Instances lower than 3 m or with fewer than 100 points are ignored.

`load_clouds()` is used by every later step that needs points; it loads each cloud once per session.

In [ ]:
INSTANCES_PATH = TABLES / 'lidar_instances.gpkg'
DENSITY_PATH = TABLES / 'point_density_per_parzelle.csv'
DTM_PATH = OUT / 'rasters' / 'dtm_lidar_after.tif'
CLIP_BEFORE = OUT / 'subset' / 'sat_before_aoi.laz'
CLIP_AFTER = OUT / 'subset' / 'sat_after_aoi.laz'
_clouds = {}


def clip_flights():
    for name, src, dst in [('sat_before', SAT_BEFORE, CLIP_BEFORE), ('sat_after', SAT_AFTER, CLIP_AFTER)]:
        if FORCE_RECOMPUTE or not dst.exists():
            t0 = time.time()
            tmp = dst.with_name(dst.stem + '_partial.laz')      # so an interrupted clip is never mistaken for a full one
            info = ftm.clip_laz_to_bounds(src, tmp, aoi)
            tmp.replace(dst)
            print(f"{name}: {info['points_out']:,} of {info['points_in']:,} points in {time.time() - t0:.0f} s")


def load_clouds(with_after=False):
    """Before-harvest cloud with height above ground ('before'); with_after adds 'after' and the 'removed' flag."""
    if 'prepared' not in _clouds:
        clip_flights()
    if 'before' not in _clouds:
        t0 = time.time()
        _clouds['before'] = ftm.load_instance_cloud(CLIP_BEFORE, instance_dim=INSTANCE_DIM)
        print(f"Loaded {len(_clouds['before']):,} points before the harvest ({time.time() - t0:.0f} s)")
    need_dtm = 'prepared' not in _clouds and (FORCE_RECOMPUTE or not DTM_PATH.exists())
    if (with_after or need_dtm) and 'after' not in _clouds:
        t0 = time.time()
        _clouds['after'] = ftm.load_instance_cloud(CLIP_AFTER)
        print(f"Loaded {len(_clouds['after']):,} points after the harvest ({time.time() - t0:.0f} s)")
    if need_dtm:
        candidates = ftm.write_ground_candidates_laz(_clouds['after'], OUT / 'subset' / 'ground_candidates_after.laz', cell_size_m=0.25)
        dtm_info = create_lidar_dtm(candidates, DTM_PATH, resolution_m=DTM_RESOLUTION_M, lidar_assumed_crs=CRS)
        print(f"DTM: {dtm_info['ground_method']}, {dtm_info['ground_point_count']:,} ground points")
    _clouds['prepared'] = True      # clips and DTM are done for this session
    for name in ['before', 'after']:
        if name in _clouds and 'h' not in _clouds[name].columns:
            ftm.add_height_above_ground(_clouds[name], DTM_PATH)
    if with_after and 'removed' not in _clouds:
        t0 = time.time()
        _clouds['removed'] = ftm.flag_removed_points(_clouds['before'], _clouds['after'], aoi, voxel_m=REMOVED_VOXEL_M)
        print(f"Removed points: {_clouds['removed'].mean():.1%} of the before-harvest cloud ({time.time() - t0:.0f} s)")
    return _clouds


if FORCE_RECOMPUTE or not (INSTANCES_PATH.exists() and DENSITY_PATH.exists()):
    clouds = load_clouds(with_after=True)
    before, after, removed = clouds['before'], clouds['after'], clouds['removed']

    # Check: lowest return per 1 m cell should sit at ~0 m in both epochs
    for label, cloud in [('before', before), ('after', after)]:
        low, _ = ftm.rasterize_max(cloud['x'].to_numpy(), cloud['y'].to_numpy(), -cloud['h'].to_numpy(), aoi, 1.0)
        low = -low[np.isfinite(low)]
        print(f"Lowest return per 1 m cell, {label}: median {np.median(low):+.2f} m, "
              f"5-95 % {np.percentile(low, 5):+.2f} .. {np.percentile(low, 95):+.2f} m")

    t0 = time.time()
    instances = ftm.summarize_instances(before, removed, crs=CRS)
    print(f"Instances: {len(instances)} trees ({time.time() - t0:.0f} s); "
          f"{(before['instance'] > 0).mean():.1%} of points belong to a tree instance")
    instances.to_file(INSTANCES_PATH, layer='instance_base', driver='GPKG')

    # Point density per Parzelle in both flights (is every Parzelle covered?)
    ncols, nrows = int(np.ceil(aoi[2] - aoi[0])), int(np.ceil(aoi[3] - aoi[1]))
    grid_transform = rasterio.transform.from_origin(aoi[0], aoi[3], 1.0, 1.0)
    density = {}
    for label, cloud in [('before', before), ('after', after)]:
        col = np.clip((cloud['x'].to_numpy() - aoi[0]).astype(np.int64), 0, ncols - 1)
        row = np.clip((aoi[3] - cloud['y'].to_numpy()).astype(np.int64), 0, nrows - 1)
        counts = np.bincount(row * ncols + col, minlength=nrows * ncols).reshape(nrows, ncols)
        density[f'pts_m2_{label}'] = [
            float(counts[geometry_mask([geom], (nrows, ncols), grid_transform, invert=True)].mean()) for geom in plots.geometry]
        density[f'empty_cells_{label}'] = [
            float((counts[geometry_mask([geom], (nrows, ncols), grid_transform, invert=True)] == 0).mean()) for geom in plots.geometry]
    pd.DataFrame({'parzelle': plots['parzelle'], **density}).to_csv(DENSITY_PATH, index=False)

    chm_before, chm_transform = ftm.rasterize_max(before['x'].to_numpy(), before['y'].to_numpy(), before['h'].to_numpy(), aoi, CHM_RESOLUTION_M)
    chm_after, _ = ftm.rasterize_max(after['x'].to_numpy(), after['y'].to_numpy(), after['h'].to_numpy(), aoi, CHM_RESOLUTION_M)
    ftm.write_raster(chm_before, chm_transform, OUT / 'rasters' / 'chm_lidar_before.tif', CRS)
    ftm.write_raster(chm_after, chm_transform, OUT / 'rasters' / 'chm_lidar_after.tif', CRS)

    (OUT / 'run_info.json').write_text(json.dumps({
        'flight_before': SAT_BEFORE.name, 'flight_after': SAT_AFTER.name, 'instance_dim': INSTANCE_DIM,
        'points_before': int(len(before)), 'points_after': int(len(after)),
        'instances': int(len(instances)), 'removed_share': round(float(removed.mean()), 4)}, indent=2))
    del before, after, removed, chm_before, chm_after
else:
    print(f"Using cached instance table {INSTANCES_PATH.name}")

instances = gpd.read_file(INSTANCES_PATH, layer='instance_base')
density = pd.read_csv(DENSITY_PATH, dtype={'parzelle': str})
with rasterio.open(OUT / 'rasters' / 'chm_lidar_before.tif') as src:
    chm = src.read(1).astype(np.float32)
    chm[chm == src.nodata] = np.nan
    chm_transform, chm_bounds = src.transform, tuple(src.bounds)
print(f"LiDAR instances in the AOI: {len(instances)}")

## 3. Orientation check of the local plot coordinates

The stem positions depend on which plot corner is the origin of the local x / y in the sheet. Per Parzelle the sheet coordinates are tried as they are and mirrored in x, in y and in both; for each variant the share of stems with a LiDAR instance base within 1 m is counted. The cell stops if the result does not agree with `Y_MIRRORED_PARZELLEN`.

This uses the LiDAR only to decide the **orientation of a whole Parzelle** (one of four options), never the position of single trees.

In [ ]:
base_index = cKDTree(instances[['x_base', 'y_base']].to_numpy())
VARIANTS = {'as in sheet': (False, False), 'x mirrored': (True, False), 'y mirrored': (False, True), 'both mirrored': (True, True)}

rows = []
for parzelle, trees in field.groupby(field['parzelle'].astype(int)):
    row = {'parzelle': parzelle, 'trees': len(trees)}
    for name, (flip_x, flip_y) in VARIANTS.items():
        variant = trees[['tree_id', 'parzelle']].assign(
            local_x=plot_width[parzelle] - trees['local_x'] if flip_x else trees['local_x'],
            local_y=plot_depth[parzelle] - trees['local_y_sheet'] if flip_y else trees['local_y_sheet'])
        utm = transform_trees_local_to_utm(variant, corners_df)
        dist, _ = base_index.query(utm[['utm_x', 'utm_y']].to_numpy())
        row[name] = round(float((dist <= 1.0).mean()), 2)
    rows.append(row)
orientation = pd.DataFrame(rows).set_index('parzelle')
orientation['best'] = orientation[list(VARIANTS)].idxmax(axis=1)
orientation.to_csv(TABLES / 'orientation_check.csv')

found = sorted(orientation.index[orientation['best'] == 'y mirrored'])
other = sorted(orientation.index[~orientation['best'].isin(['as in sheet', 'y mirrored'])])
assert not other, f"Parzellen {other} fit best with an orientation this notebook does not handle"
assert found == sorted(mirrored_parzellen), (
    f"Orientation check finds y mirrored in Parzellen {found}, but Y_MIRRORED_PARZELLEN gives {mirrored_parzellen}. "
    "Update the setting in modules/notebook_common.py and re-run from section 1.")
print(f"Sheet orientation is best in {int((orientation['best'] == 'as in sheet').sum())} of {len(orientation)} Parzellen; "
      f"y mirrored in {found}, as set.")

## 4. Matching field stems to LiDAR instances

Three assignments with the same function:
- `plain`: distance only. Used wherever the harvest record is compared with the LiDAR, so that comparison stays independent of the matching.
- `matches`: the final assignment, with the harvest penalty. This gives the detection class.
- `control`: all stems shifted by 5 m to the north, east, south and west. Whatever these non-existent trees still match is the chance level.

In [ ]:
stems = field[['tree_id', 'utm_x', 'utm_y', 'removed_2026']]      # no height columns: heights play no role

def match(stem_table, removal_penalty_m=REMOVAL_PENALTY_M):
    return ftm.match_field_trees_to_instances(
        stem_table, instances,
        max_distance_m=MAX_MATCH_DISTANCE_M, removed_threshold=REMOVED_THRESHOLD,
        removal_penalty_m=removal_penalty_m, high_confidence_distance_m=HIGH_CONFIDENCE_DISTANCE_M)

plain = field[['tree_id', 'block', 'removed_2026']].merge(match(stems, removal_penalty_m=0.0), on='tree_id')
matches = match(stems)
assert not matches['instance'].dropna().duplicated().any(), "An instance was assigned to more than one field tree"

result = field.merge(matches.drop(columns=['field_height', 'height_gate_ok']), on='tree_id', how='left')
result['detection'] = result['confidence'].map({'high': 'detected', 'check': 'uncertain', 'unmatched': 'not detected'})
result['detection'] = pd.Categorical(result['detection'], categories=CLASSES)
result['uncertain_reason'] = np.select(
    [result['detection'] != 'uncertain',
     (result['base_dist_m'] > HIGH_CONFIDENCE_DISTANCE_M) & result['removal_consistent'].eq(False),
     result['base_dist_m'] > HIGH_CONFIDENCE_DISTANCE_M],
    ['', 'far and harvest evidence disagrees', 'base 1.5-3 m away'], default='harvest evidence disagrees')
print(f"All {len(result)} standing trees: {result['detection'].value_counts().to_dict()}")

# Chance control
control = []
for dx, dy, label in [(0, 1, 'north'), (1, 0, 'east'), (0, -1, 'south'), (-1, 0, 'west')]:
    shifted = stems.assign(utm_x=stems['utm_x'] + dx * CONTROL_SHIFT_M, utm_y=stems['utm_y'] + dy * CONTROL_SHIFT_M)
    control.append(match(shifted).assign(shift=label))
control = pd.concat(control, ignore_index=True)
control[['tree_id', 'shift', 'instance', 'base_dist_m', 'removal_consistent', 'confidence']].to_csv(
    TABLES / 'chance_control_matches.csv', index=False)

# Distance-only assignment with the record and the LiDAR state of the instance at the stem
felled = plain[['tree_id', 'block', 'removed_2026', 'instance', 'base_dist_m', 'removed_fraction']].merge(
    field[['tree_id', 'parzelle', 'species', 'dbh_mm', 'remark']], on='tree_id')
felled['parzelle'] = felled['parzelle'].astype(int)
felled['record'] = felled['removed_2026'].map(rs.RECORD)
felled['lidar'] = pd.Categorical(rs.lidar_state(felled['removed_fraction'], felled['base_dist_m'] <= HIGH_CONFIDENCE_DISTANCE_M),
                                 categories=rs.LIDAR_STATES)
felled.to_csv(TABLES / 'match_distance_only.csv', index=False)

# LiDAR side: instances with their base inside a Parzelle, and the field tree they are matched to
inst = gpd.sjoin(instances, plots[['parzelle', 'geometry']], predicate='within', how='left').drop(columns='index_right')
inst = inst[~inst.index.duplicated()].copy()
matched_tree = result.dropna(subset=['instance']).set_index(result.dropna(subset=['instance'])['instance'].astype(int))
inst['tree_id'] = inst['instance'].map(matched_tree['tree_id'])
inst['detection'] = inst['instance'].map(matched_tree['detection'].astype(str))
inst['in_plot'] = inst['parzelle'].notna()
inst['gone_after_harvest'] = inst['removed_fraction'] > REMOVED_THRESHOLD
in_plot = inst[inst['in_plot']]
per_plot = rs.detection_per_parzelle(result, in_plot, corner_rmse)
per_plot.to_csv(TABLES / 'detection_per_parzelle.csv')

In [ ]:
# Export: table and GeoPackage (layers field_trees, lidar_instances, stem_to_base, parzellen)
from shapely.geometry import LineString

table = result.copy()
table['instance'] = table['instance'].astype('Int64')
table['runner_up_instance'] = table['runner_up_instance'].astype('Int64')
table['removal_consistent'] = table['removal_consistent'].astype('boolean')
table['detection'] = table['detection'].astype(str)
table['dbh_class'] = table['dbh_class'].astype(str)
table = table.drop(columns=['confidence', 'method'])
DETECTION_PATH = TABLES / 'field_trees_detection.csv'
table.to_csv(DETECTION_PATH, index=False)

GPKG_PATH = TABLES / 'detection_all_trees.gpkg'
if GPKG_PATH.exists():
    GPKG_PATH.unlink()
gpd.GeoDataFrame(table, geometry=gpd.points_from_xy(table['utm_x'], table['utm_y']), crs=CRS).to_file(GPKG_PATH, layer='field_trees', driver='GPKG')
inst.to_file(GPKG_PATH, layer='lidar_instances', driver='GPKG')
linked = table[table['x_base'].notna()]
gpd.GeoDataFrame(linked[['tree_id', 'instance', 'detection', 'base_dist_m']],
                 geometry=[LineString([(r.utm_x, r.utm_y), (r.x_base, r.y_base)]) for r in linked.itertuples()],
                 crs=CRS).to_file(GPKG_PATH, layer='stem_to_base', driver='GPKG')
plots.merge(per_plot.reset_index().assign(parzelle=lambda d: d['parzelle'].astype(str)), on='parzelle').to_file(GPKG_PATH, layer='parzellen', driver='GPKG')
print(f"Table:      {DETECTION_PATH}  ({len(table)} trees)")
print(f"GeoPackage: {GPKG_PATH}")

## 5. Stand structure at every tree

Per tree: DBH relative to the mean of its Parzelle, distance to the nearest field stem and to the nearest larger one, a competition index (Hegyi: sum over the neighbours within 5 m of neighbour DBH / own DBH / distance), and the canopy height at the stem from the LiDAR CHM against the height a tree of that DBH should have (height curve fitted to the detected trees). The field height is not used; only a small part of the trees has one.

The `case` column is the first, coarser reading of why a tree was missed (kept as it was); section 7 gives the cause that also uses the stem.

Per Parzelle: canopy top height, roughness and gap share from the CHM.

In [ ]:
from scipy.optimize import curve_fit

struct = result[['tree_id', 'parzelle', 'species', 'removed_2026', 'utm_x', 'utm_y', 'detection', 'uncertain_reason',
                 'base_dist_m', 'removed_fraction', 'runner_up_instance', 'lidar_height']].copy()
struct['parzelle'] = struct['parzelle'].astype(int)
struct['dbh_mm'] = result['dbh_mm'].where(result['dbh_mm'].between(*DBH_PLAUSIBLE_MM))
struct['rel_dbh'] = struct['dbh_mm'] / struct.groupby('parzelle')['dbh_mm'].transform('mean')

# Neighbourhood from the field stems of the whole site, so neighbours across a Parzelle border count as well
stem_xy = struct[['utm_x', 'utm_y']].to_numpy()
stem_index = cKDTree(stem_xy)
nn_dist, _ = stem_index.query(stem_xy, k=2)
struct['nn_dist_m'] = nn_dist[:, 1]
dbh_filled = struct['dbh_mm'].fillna(struct['dbh_mm'].median()).to_numpy()
hegyi, larger_dist = [], []
for i, neighbours in enumerate(stem_index.query_ball_point(stem_xy, COMPETITION_RADIUS_M)):
    neighbours = np.array([j for j in neighbours if j != i], dtype=int)
    dist = np.hypot(*(stem_xy[neighbours] - stem_xy[i]).T) if len(neighbours) else np.array([])
    hegyi.append(float(np.sum(dbh_filled[neighbours] / dbh_filled[i] / np.maximum(dist, 0.3))))
    larger = dist[dbh_filled[neighbours] > dbh_filled[i]]
    larger_dist.append(larger.min() if len(larger) else np.nan)
struct['hegyi'] = hegyi                           # high = small tree with large, close neighbours
struct['larger_neighbour_dist_m'] = larger_dist   # NaN: no larger tree within the competition radius

# Canopy height at the stem, and the height a tree of this DBH is expected to have
rows, cols = rasterio.transform.rowcol(chm_transform, struct['utm_x'].to_numpy(), struct['utm_y'].to_numpy())
k = int(round(CHM_STEM_RADIUS_M / CHM_RESOLUTION_M))
struct['chm_at_stem_m'] = [np.nanmax(chm[max(r - k, 0):r + k + 1, max(c - k, 0):c + k + 1]) for r, c in zip(rows, cols)]

def height_curve(dbh, a, b, c):
    return 1.3 + a * (1 - np.exp(-b * dbh)) ** c

fit = struct[(struct['detection'] == 'detected') & struct['dbh_mm'].notna() & struct['lidar_height'].notna()]
curve_params, _ = curve_fit(height_curve, fit['dbh_mm'], fit['lidar_height'], p0=[35, 0.005, 1], maxfev=20000)
struct['expected_height_m'] = height_curve(struct['dbh_mm'].fillna(struct['dbh_mm'].median()), *curve_params)
struct['canopy_deficit_m'] = struct['expected_height_m'] - struct['chm_at_stem_m']
print(f"Height curve from the detected trees (LiDAR height over DBH): residual SD "
      f"{(fit['lidar_height'] - height_curve(fit['dbh_mm'], *curve_params)).std():.1f} m, n = {len(fit)}")

struct['case'] = ''
nd = struct['detection'] == 'not detected'
struct.loc[nd, 'case'] = np.select(
    [struct.loc[nd, 'canopy_deficit_m'] > NO_CANOPY_DEFICIT_M, struct.loc[nd, 'runner_up_instance'].notna()],
    ['no canopy at the stem', 'instance taken by a neighbour'], default='no instance base within 3 m')
unc = struct['detection'] == 'uncertain'
u = struct[unc]
struct.loc[unc, 'case'] = np.select(
    [u['base_dist_m'] > HIGH_CONFIDENCE_DISTANCE_M,
     ~u['removed_2026'] & (u['removed_fraction'] > GONE_ABOVE),
     u['removed_2026'] & (u['removed_fraction'] < STANDING_BELOW)],
    ['base 1.5-3 m away', 'not marked, instance gone', 'marked, instance still standing'], default='instance partly gone')
struct.drop(columns=['utm_x', 'utm_y']).to_csv(TABLES / 'structure_per_tree.csv', index=False)

# Canopy per Parzelle, from the CHM
plot_geom = plots.assign(parzelle=plots['parzelle'].astype(int)).set_index('parzelle').sort_index()
canopy = []
for parzelle, geom in plot_geom.geometry.items():
    values = chm[geometry_mask([geom], chm.shape, chm_transform, invert=True)]
    values = values[np.isfinite(values)]
    canopy.append({'parzelle': parzelle, 'area_m2': geom.area, 'canopy_top_height_m': np.percentile(values, 95),
                   'canopy_height_sd_m': values.std(), 'gap_share': (values < 5).mean()})      # gap: canopy below 5 m
canopy = pd.DataFrame(canopy).set_index('parzelle')
canopy.to_csv(TABLES / 'canopy_per_parzelle.csv')
print(f"Wrote structure_per_tree.csv ({len(struct)} trees) and canopy_per_parzelle.csv ({len(canopy)} Parzellen)")

## 6. The stem at every field position (cached)

For every field tree, detected or not, the stem is searched at the **field position** and followed upwards through eight thin slices from 1.5 to 8.5 m (`ftm.stem_profiles`, a circle fit per slice). This gives
- whether the LiDAR shows a stem at the position at all, and in how many slices,
- **which instance the stem points belong to**: the tree's own instance, the instance of another field tree, an instance without a field tree, or no instance,
- stem form: lean, sweep and taper.

`ftm.column_structure` describes the point column around each position: how many separate canopy layers it has (groups of returns separated by at least 2 m without any return) and how many returns hit the lower stem.

In [ ]:
STEM_FORM_PATH = TABLES / 'stem_form_per_tree.csv'
COLUMN_PATH = TABLES / 'column_structure_per_tree.csv'
field_positions = field[['tree_id']].assign(x=field['utm_x'], y=field['utm_y'])

if FORCE_RECOMPUTE or not STEM_FORM_PATH.exists():
    cloud = load_clouds()['before']
    t0 = time.time()
    stem_zone = cloud[(cloud['h'] > 1.0) & (cloud['h'] < 9.0)]
    # Start at the field position for every tree, detected or not, so all classes are treated alike
    ftm.stem_profiles(stem_zone, field_positions).to_csv(STEM_FORM_PATH, index=False)
    print(f"Stem profiles ({time.time() - t0:.0f} s)")
    del stem_zone
else:
    print(f"Using cached {STEM_FORM_PATH.name}")

if FORCE_RECOMPUTE or not COLUMN_PATH.exists():
    t0 = time.time()
    ftm.column_structure(load_clouds()['before'], field_positions).to_csv(COLUMN_PATH, index=False)
    print(f"Column structure ({time.time() - t0:.0f} s)")
else:
    print(f"Using cached {COLUMN_PATH.name}")

stem_form = pd.read_csv(STEM_FORM_PATH, dtype={'tree_id': str})
column = pd.read_csv(COLUMN_PATH, dtype={'tree_id': str})
print(f"Stem found at the field position: {(stem_form['stem_slices'] > 0).sum()} of {len(stem_form)} trees")

In [ ]:
# One table with everything known about a tree
profile = struct.merge(result[['tree_id', 'instance', 'local_x', 'local_y', 'block', 'has_height', 'height_vertex', 'height_massband',
                               'height_standing_sheet', 'dbh_2020_mm', 'dbh_year']], on='tree_id', how='left').merge(stem_form, on='tree_id', how='left')
assert profile['tree_id'].equals(struct['tree_id'])

# Field height where one was measured: felled length first, else the height taken on the standing tree
standing = profile['height_standing_sheet'].where(profile['height_standing_sheet'] <= STANDING_HEIGHT_DM_ABOVE,
                                                   profile['height_standing_sheet'] / 10)
profile['field_height_m'] = profile[['height_vertex', 'height_massband']].mean(axis=1).fillna(standing)

# Diameter growth since the 2020 inventory
dbh_2020 = profile['dbh_2020_mm'].where(profile['dbh_2020_mm'].between(*DBH_PLAUSIBLE_MM))
profile['dbh_increment_mm_yr'] = (profile['dbh_mm'] - dbh_2020) / (profile['dbh_year'] - 2020)

# Neighbourhood and position
profile['stems_within_5m'] = stem_index.query_ball_point(stem_xy, COMPETITION_RADIUS_M, return_length=True) - 1
profile['parzelle_stems_per_ha'] = profile['parzelle'].map(struct.groupby('parzelle').size() / (plot_geom.area / 1e4))
width, depth = profile['parzelle'].map(plot_width), profile['parzelle'].map(plot_depth)
profile['edge_dist_m'] = np.minimum.reduce([profile['local_x'], width - profile['local_x'], profile['local_y'], depth - profile['local_y']]).clip(min=0)

# Which instance do the LiDAR points of the stem at the field position belong to?
STEM_OWNERS = ['own instance', 'instance of another field tree', 'instance without a field tree', 'not in any instance', 'no stem at the position']
own_instance = pd.to_numeric(profile['instance'], errors='coerce').to_numpy(dtype=float, na_value=np.nan)
other_tree = profile['stem_instance'].isin(matched_tree.index)
profile['stem_owner'] = pd.Categorical(np.select(
    [profile['stem_slices'] == 0, profile['stem_instance'] == 0, profile['stem_instance'].to_numpy() == own_instance, other_tree],
    [STEM_OWNERS[4], STEM_OWNERS[3], STEM_OWNERS[0], STEM_OWNERS[1]], default=STEM_OWNERS[2]), categories=STEM_OWNERS)

SPECIES_GROUPS = {'Fichte': 'spruce', 'Europäische Lärche': 'other conifers', 'Douglasie': 'other conifers', 'Kiefer': 'other conifers',
                  'Buche': 'broadleaves', 'Sonstige Laubhölzer': 'broadleaves'}
profile['species_group'] = profile['species'].map(SPECIES_GROUPS).fillna('other')

profile_out = profile.drop(columns=['utm_x', 'utm_y', 'height_vertex', 'height_massband', 'height_standing_sheet'])
profile_out.to_csv(TABLES / 'tree_profile_all.csv', index=False)
missed = profile_out[profile_out['detection'] != 'detected'].sort_values(['detection', 'case', 'parzelle', 'tree_id'])
missed.to_csv(TABLES / 'missed_trees_profile.csv', index=False)
print(f"tree_profile_all.csv ({len(profile_out)} trees), missed_trees_profile.csv ({len(missed)} not detected or uncertain trees)")

## 7. Why a tree is uncertain or not detected

One cause per uncertain or not detected tree, decided by rules on the evidence collected above (`modules/match_causes.py`). The rules, in the order they are applied:

**Not detected** (no free instance base within 3 m)
1. canopy at the position more than 6 m below the expected height → *no tree at the position on the flight day* (field record or timing)
2. stem found, its points belong to the instance of another field tree → *merged with a neighbour* (SegmentAnyTree)
3. stem found, its points belong to an instance without a field tree → *own instance, base far from the stem* (SegmentAnyTree)
4. stem found, its points belong to no instance → *stem visible, not segmented* (SegmentAnyTree)
5. no stem found, canopy present → *stem not visible in the LiDAR* (LiDAR coverage)

**Uncertain** (an instance is assigned)
1. the stem at the field position belongs to the instance of another field tree → *merged with a neighbour* (SegmentAnyTree)
2. base 1.5–3 m away, stem at the position belongs to an instance → *own instance, base far from the stem* (SegmentAnyTree)
3. base 1.5–3 m away otherwise → *instance 1.5–3 m away, not confirmed by a stem* (unclear)
4. base within 1.5 m, instance clearly gone or clearly standing, record says the opposite → *tree found, harvest record disagrees* (field record or timing)
5. base within 1.5 m, 20–80 % of the instance gone → *tree found, instance partly gone* (SegmentAnyTree)

The table also holds the stand conditions of **every** tree (also the detected ones), so the results notebook can compare them: share of larger neighbours within 5 m, number and height spread of the LiDAR instances within 10 m, canopy layers and stem hits of the point column.

These rules are a reading of the evidence, not a measurement. Section 10 writes a QA figure for every uncertain and not detected tree to check them.

In [ ]:
causes = match_causes.classify_match_cause(profile)
conditions = match_causes.stand_conditions(result[['tree_id', 'utm_x', 'utm_y', 'dbh_mm']], instances)
causes = causes.merge(conditions, on='tree_id').merge(column, on='tree_id', how='left')
# positive: the canopy at the stem is higher than this tree is expected to be, i.e. it stands under larger trees
causes['overtopped_by_m'] = (struct['chm_at_stem_m'] - struct['expected_height_m']).to_numpy()
causes.to_csv(TABLES / 'match_causes.csv', index=False)

missed_causes = causes[causes['cause'] != '']
assert len(missed_causes) == int((result['detection'] != 'detected').sum())
print(pd.crosstab(missed_causes['cause'], missed_causes['detection'], margins=True).to_string())
print("\nAttributed to:", missed_causes['attributed_to'].value_counts().to_dict())

## 8. Photogrammetry height and LiDAR DBH of every tree (cached)

- **Photogrammetry height**: maximum of the photogrammetric surface model (25 Feb 2026) within 1 m of the LiDAR top, above the LiDAR ground model. `visible_from_above` says whether the LiDAR top lies within 1 m of the canopy surface around it; only such trees can be measured in a surface model.
- **Independent photogrammetric tree tops**: local maxima of the photogrammetric nDSM; `photo_detected` = one of them lies within 1.5 m of the LiDAR top.
- **Ground offset**: photogrammetric surface minus LiDAR ground on open ground, before and after the harvest. Any offset appears one-to-one as bias in the photogrammetry heights.
- **LiDAR DBH**: circle fit (RANSAC, then least squares) on the 1.05–1.55 m slice at the LiDAR stem base, or at the field position for trees without an instance. `dbh_quality` is `ok`, `weak` or `no_fit`. An ellipse is fitted as well; its perimeter-equivalent diameter is what a girth tape measures.

In [ ]:
VALUES_PATH = TABLES / 'height_dbh_all_trees.csv'
NDSM_BEFORE = OUT / 'rasters' / 'ndsm_photo_before.tif'
NDSM_AFTER = OUT / 'rasters' / 'ndsm_photo_after.tif'
OFFSET_PATH = TABLES / 'photo_ground_offset.csv'

if FORCE_RECOMPUTE or not (VALUES_PATH.exists() and OFFSET_PATH.exists() and NDSM_AFTER.exists()):
    for dsm, ndsm, clip in [(DSM_BEFORE, NDSM_BEFORE, 'dsm_before_aoi.tif'), (DSM_AFTER, NDSM_AFTER, 'dsm_after_aoi.tif')]:
        if FORCE_RECOMPUTE or not ndsm.exists():
            create_ndsm(subset_raster(dsm, chm_bounds, OUT / 'subset' / clip), DTM_PATH, ndsm, clip_negative_to_zero=False)
    with rasterio.open(OUT / 'rasters' / 'chm_lidar_after.tif') as src:
        chm_after = src.read(1).astype(np.float32)
        chm_after[chm_after == src.nodata] = np.nan
    pd.DataFrame([{'epoch': 'before', **ftm.open_ground_offset(chm, chm_transform, NDSM_BEFORE)},
                  {'epoch': 'after', **ftm.open_ground_offset(chm_after, chm_transform, NDSM_AFTER)}]).to_csv(OFFSET_PATH, index=False)
    del chm_after

    with_top = result[result['x_top'].notna()]
    photo = extract_ndsm_tree_heights(
        with_top[['tree_id']].assign(x=with_top['utm_x'], y=with_top['utm_y']), NDSM_BEFORE,
        top_coordinates=with_top[['tree_id', 'x_top', 'y_top']], method='raster_max', search_radius_m=PHOTO_RADIUS_M)
    values = result[['tree_id']].merge(
        photo[['tree_id', 'photogrammetry_height']].rename(columns={'photogrammetry_height': 'photo_height'}), on='tree_id', how='left')
    values['visible_from_above'] = ftm.visible_from_above(
        chm, chm_transform, result['x_top'].to_numpy(dtype=float), result['y_top'].to_numpy(dtype=float),
        result['lidar_height'].to_numpy(dtype=float), radius_m=1.0, tolerance_m=VISIBLE_TOLERANCE_M)

    # Independent tree-top detection on the photogrammetric nDSM
    with rasterio.open(NDSM_BEFORE) as src:
        ndsm_res = src.res[0]
    photo_tops = detect_ndsm_tree_tops(NDSM_BEFORE, kernel_size=int(round(1.5 / ndsm_res)) | 1, min_height_m=5.0,
                                       min_separation_m=1.5, smoothing_sigma=0.25 / ndsm_res)
    photo_tops.to_file(TABLES / 'photo_tree_tops.gpkg', layer='photo_tops', driver='GPKG')
    dist, _ = cKDTree(photo_tops[['x_top', 'y_top']].to_numpy()).query(result[['x_top', 'y_top']].fillna(1e12).to_numpy())
    values['photo_top_dist_m'] = np.where(result['x_top'].notna(), dist, np.nan)
    values['photo_detected'] = values['photo_top_dist_m'] <= PHOTO_DETECT_DISTANCE_M

    # DBH: look for the stem at the LiDAR base, or at the field position if the tree has no instance
    use_lidar = result['x_base'].notna()
    centres = pd.DataFrame({'tree_id': result['tree_id'],
                            'x': np.where(use_lidar, result['x_base'], result['utm_x']),
                            'y': np.where(use_lidar, result['y_base'], result['utm_y'])})
    t0 = time.time()
    dbh = ftm.estimate_dbh(load_clouds()['before'], centres, breast_height_m=DBH_BREAST_HEIGHT_M,
                           slice_half_height_m=DBH_SLICE_HALF_M, search_radius_m=DBH_SEARCH_RADIUS_M)
    print(f"DBH fits for {len(dbh)} trees ({time.time() - t0:.0f} s)")
    values['dbh_centre_from'] = np.where(use_lidar, 'LiDAR stem base', 'field position')
    values = values.merge(dbh, on='tree_id', how='left')
    values.to_csv(VALUES_PATH, index=False)
else:
    print(f"Using cached {VALUES_PATH.name}")

values = pd.read_csv(VALUES_PATH, dtype={'tree_id': str})
print(f"Photogrammetry height for {values['photo_height'].notna().sum()} trees, "
      f"{int(values['visible_from_above'].sum())} visible from above; circle fit {values['dbh_quality'].value_counts().to_dict()}")
print(pd.read_csv(OFFSET_PATH).round(3).to_string(index=False))

## 9. Result tiers: automatic, manually picked, hand-segmented

The detection class of section 4 never uses a field height. For the trees that have one, three more things are recorded here, each as its own tier so that SegmentAnyTree numbers never contain manual work:

- **Height gate** (flag only): LiDAR and field height differ by more than 25 %.
- **Column profile fallback**: for trees without an instance or with a failed gate, the top of the lowest continuous point layer above the stem. If a tree failed the gate and has no separate layer, it is `unresolved`: merged into a neighbour, no LiDAR height.
- **Manual corrections**, applied on every run:
  - another SegmentAnyTree instance fits better: add a row to `tables/match_overrides.csv` (columns `tree_id,instance`; instance `0` = no LiDAR tree)
  - no instance fits: cut the tree out by hand and save it as `manual_trees/tree_<Parzelle>_<Nr>.laz`

`auto_height`, `auto_confidence` and `auto_method` are the automatic result and do not change when a tree is corrected by hand.

In [ ]:
TIERS_AUTO_PATH = TABLES / 'match_tiers_auto.csv'
TIER_COLUMNS = ['tree_id', 'instance', 'lidar_height', 'sat_height', 'x_top', 'y_top', 'x_base', 'y_base', 'field_height',
                'height_gate_ok', 'confidence', 'method']

if FORCE_RECOMPUTE or not TIERS_AUTO_PATH.exists():
    clouds = load_clouds(with_after=True)
    auto = ftm.add_height_gate(matches, field, HEIGHT_GATE_REL)
    auto = ftm.add_column_profile_fallback(auto, field, clouds['before'], clouds['removed'])
    auto[TIER_COLUMNS].to_csv(TIERS_AUTO_PATH, index=False)
else:
    print(f"Using cached {TIERS_AUTO_PATH.name}")

tiers = pd.read_csv(TIERS_AUTO_PATH, dtype={'tree_id': str})
tiers['instance'] = tiers['instance'].astype('Int64')
assert tiers['tree_id'].equals(result['tree_id']), "match_tiers_auto.csv belongs to another matching: set FORCE_RECOMPUTE = True"
tiers['auto_height'], tiers['auto_confidence'], tiers['auto_method'] = tiers['lidar_height'], tiers['confidence'], tiers['method']

OVERRIDES_PATH = TABLES / 'match_overrides.csv'
if OVERRIDES_PATH.exists():
    overrides = pd.read_csv(OVERRIDES_PATH, dtype={'tree_id': str})
    tiers = ftm.apply_match_overrides(tiers, instances, overrides)
    print(f"Applied {len(overrides)} manual overrides from {OVERRIDES_PATH.name}")
manual, manual_points = ftm.load_manual_segments(OUT / 'manual_trees', DTM_PATH)
if len(manual):
    tiers = ftm.apply_manual_segments(tiers, manual)
    print(f"Applied {len(manual)} hand-segmented trees: {manual['tree_id'].tolist()}")
tiers.to_csv(TABLES / 'match_tiers.csv', index=False)

sample_tiers = tiers[tiers['tree_id'].isin(field.loc[field['has_height'], 'tree_id'])]
print(f"All trees, automatic: {tiers['auto_confidence'].value_counts().to_dict()}")
print(f"Sample trees, automatic: {sample_tiers['auto_confidence'].value_counts().to_dict()}; method {sample_tiers['auto_method'].value_counts().to_dict()}")

## 10. QA figures and inspection clips

One figure per field tree of the classes in `QA_CLASSES` and per sample tree: orthoimage before and after the harvest, LiDAR CHM before, and a 3 m wide vertical slab through the field stem (red dotted line) towards the LiDAR top. Red = assigned instance, orange = other points gone after the harvest, grey = everything else. Figures that already exist are not redrawn.

- `figures/qa_not_detected/`, `figures/qa_uncertain/`: the trees to check the causes of section 7 against
- `figures/qa_sample_trees/`: the trees with a felled length, plus the CHM before / after the harvest
- `figures/dbh_slices_sample_trees.png`: breast-height slice of each sample tree with the fitted circle (red), ellipse (green) and field DBH (blue, dashed)

**Inspection clips** (`WRITE_INSPECTION_CLIPS = True`): a cylinder of the before- and after-harvest cloud around each tree that needs a look, in `inspect/`. In CloudCompare, colour `tree_<id>_before.laz` by the instance field; `matched` = 1 marks the assigned instance, `removed` = 1 the points gone after the harvest. Then either add a row to `tables/match_overrides.csv` or save a hand-cut tree to `manual_trees/`, and re-run from section 9.

In [ ]:
qa = result.merge(tiers[['tree_id', 'auto_confidence', 'sat_height']], on='tree_id')
qa['confidence'], qa['method'] = qa['detection'].astype(str), 'sat_instance'
jobs = [(name, OUT / 'figures' / f"qa_{name.replace(' ', '_')}", qa[qa['detection'] == name]) for name in QA_CLASSES]
jobs.append(('sample trees', OUT / 'figures' / 'qa_sample_trees', qa[qa['has_height']]))
todo = [(folder, tree) for _, folder, trees in jobs
        for _, tree in trees.sort_values('dbh_mm', ascending=False, na_position='last').iterrows()
        if FORCE_RECOMPUTE or not (folder / f"tree_{tree['tree_id']}.png").exists()]
SLICES_PATH = OUT / 'figures' / 'dbh_slices_sample_trees.png'
print({name: f"{len(trees)} trees" for name, _, trees in jobs}, f"-> {len(todo)} figures to draw")

if todo or WRITE_INSPECTION_CLIPS or FORCE_RECOMPUTE or not SLICES_PATH.exists():
    clouds = load_clouds(with_after=True)
    before, removed = clouds['before'], clouds['removed']
    with rasterio.open(OUT / 'rasters' / 'chm_lidar_after.tif') as src:
        chm_after = src.read(1).astype(np.float32)
        chm_after[chm_after == src.nodata] = np.nan

    def chm_crop(raster, bounds):
        col0, row0 = ~chm_transform * (bounds[0], bounds[3])
        col1, row1 = ~chm_transform * (bounds[2], bounds[1])
        return raster[max(int(row0), 0):int(row1) + 1, max(int(col0), 0):int(col1) + 1]

    t0 = time.time()
    for folder, tree in todo:
        folder.mkdir(parents=True, exist_ok=True)
        fig = ftm.plot_match_qa(tree, before, removed, chm, chm_transform, ORTHO_BEFORE, ORTHO_AFTER,
                                output_path=folder / f"tree_{tree['tree_id']}.png", manual_points=manual_points.get(tree['tree_id']))
        plt.close(fig)
        if folder.name != 'qa_sample_trees':
            continue
        # CHM before and after the harvest, and the height loss: a felled tree shows up as a crown-shaped patch
        sx, sy = float(tree['utm_x']), float(tree['utm_y'])
        bounds = (sx - 10.0, sy - 10.0, sx + 10.0, sy + 10.0)
        crop_before, crop_after = chm_crop(chm, bounds), chm_crop(chm_after, bounds)
        fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
        for ax, (data, title, cmap, vmax) in zip(axes, [(crop_before, f"LiDAR CHM before ({BEFORE_FLIGHT})", 'viridis', 38),
                                                        (crop_after, 'LiDAR CHM after (2026-03-24)', 'viridis', 38),
                                                        (crop_before - crop_after, 'Height loss (before - after)', 'magma', 35)]):
            im = ax.imshow(data, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], cmap=cmap, vmin=0, vmax=vmax)
            fig.colorbar(im, ax=ax, fraction=0.046, label='m')
            ax.plot(sx, sy, 'x', color='red', markersize=11, markeredgewidth=2.5, label='field stem')
            if pd.notna(tree['x_top']):
                ax.plot(tree['x_base'], tree['y_base'], '+', color='cyan', markersize=11, markeredgewidth=2, label='LiDAR base')
                ax.plot(tree['x_top'], tree['y_top'], '^', color='yellow', markeredgecolor='black', markersize=9, label='LiDAR top')
            ax.set_title(title)
            ax.set_xticks([])
            ax.set_yticks([])
        axes[0].legend(loc='upper left', fontsize=8)
        fig.suptitle(f"Tree {tree['tree_id']}  |  {tree['detection']}  |  removed 2026: {bool(tree['removed_2026'])}, "
                     f"LiDAR removed fraction {tree['removed_fraction']:.2f}", fontsize=11)
        fig.tight_layout()
        fig.savefig(folder / f"tree_{tree['tree_id']}_chm_before_after.png", dpi=110, bbox_inches='tight')
        plt.close(fig)
    print(f"Drew {len(todo)} QA figures ({time.time() - t0:.0f} s)")

    # Breast-height slices of the sample trees: circle (red), ellipse (green), field DBH (blue, dashed)
    sample = result[result['has_height']]
    use_lidar = sample['x_base'].notna()
    centres = pd.DataFrame({'tree_id': sample['tree_id'], 'x': np.where(use_lidar, sample['x_base'], sample['utm_x']),
                            'y': np.where(use_lidar, sample['y_base'], sample['utm_y'])})
    fits, slices = ftm.estimate_dbh(before, centres, breast_height_m=DBH_BREAST_HEIGHT_M, slice_half_height_m=DBH_SLICE_HALF_M,
                                    search_radius_m=DBH_SEARCH_RADIUS_M, return_slices=True)
    fits = fits.merge(centres, on='tree_id').merge(sample[['tree_id', 'dbh_mm']], on='tree_id').sort_values('dbh_mm').reset_index(drop=True)
    ncols = 6
    nrows = int(np.ceil(len(fits) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.5 * nrows))
    theta = np.linspace(0, 2 * np.pi, 200)
    for ax, (_, t) in zip(axes.ravel(), fits.iterrows()):
        xy = slices[t['tree_id']]
        cx, cy = (t['x_stem'], t['y_stem']) if pd.notna(t['x_stem']) else (t['x'], t['y'])
        if len(xy):
            ax.scatter(xy[:, 0] - cx, xy[:, 1] - cy, s=4, color='0.4')
        if pd.notna(t['dbh_lidar_mm']):
            r = t['dbh_lidar_mm'] / 2000
            ax.plot(r * np.cos(theta), r * np.sin(theta), color='red', linewidth=1.2)
        if pd.notna(t.get('dbh_ellipse_mm')):
            a, b, rot = t['ellipse_a_mm'] / 1000, t['ellipse_b_mm'] / 1000, t['ellipse_theta']
            ex, ey = a * np.cos(theta), b * np.sin(theta)
            ax.plot(t['ellipse_cx'] - cx + ex * np.cos(rot) - ey * np.sin(rot),
                    t['ellipse_cy'] - cy + ex * np.sin(rot) + ey * np.cos(rot), color='tab:green', linewidth=1.1)
        if pd.notna(t['dbh_mm']):
            r = t['dbh_mm'] / 2000
            ax.plot(r * np.cos(theta), r * np.sin(theta), color='tab:blue', linewidth=1, linestyle='--')
        circle_txt = f"{t['dbh_lidar_mm']:.0f}" if pd.notna(t['dbh_lidar_mm']) else 'n/a'
        ax.set_title(f"{t['tree_id']}  field {t['dbh_mm']:.0f} mm\ncircle {circle_txt} mm\n{t['dbh_quality']}, arc {t['arc_coverage']:.0%}", fontsize=8)
        ax.set_xlim(-0.6, 0.6)
        ax.set_ylim(-0.6, 0.6)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.ravel()[len(fits):]:
        ax.axis('off')
    fig.suptitle(f'Breast-height slices ({DBH_BREAST_HEIGHT_M - DBH_SLICE_HALF_M:.2f}-{DBH_BREAST_HEIGHT_M + DBH_SLICE_HALF_M:.2f} m): '
                 'red = circle fit, green = ellipse fit, blue dashed = field DBH; panels 1.2 x 1.2 m', fontsize=11)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(SLICES_PATH, dpi=130, bbox_inches='tight')
    plt.close(fig)

    if WRITE_INSPECTION_CLIPS:
        sample_qa = qa[qa['has_height']]
        low = sample_qa['height_vertex'].fillna(sample_qa['height_massband']) < 20
        inspect_ids = sample_qa.loc[(sample_qa['auto_confidence'] != 'high') | low, 'tree_id'].tolist()
        inspect_ids += [t for t in INSPECT_EXTRA if t not in inspect_ids and t in set(qa['tree_id'])]
        t0 = time.time()
        clips = ftm.export_inspection_clips(qa, before, removed, clouds['after'], CLIP_BEFORE, CLIP_AFTER,
                                            OUT / 'inspect', inspect_ids, radius_m=5.0)
        print(f"Wrote clips for {len(clips)} trees to {OUT / 'inspect'} ({time.time() - t0:.0f} s)")
    del before, removed, chm_after
else:
    print("All QA figures exist.")

## 11. Matching with DBH and height as support

Does it help to use more than the stem position? Three assignments with `ftm.match_stems_with_attributes`, all one-to-one within 3 m:

| | Algorithm | Cost of a pair (field stem, instance) |
|---|---|---|
| 1 | position | distance from the field stem to the instance base |
| 2 | position + DBH | as 1, plus a penalty when LiDAR DBH and field DBH differ by more than 100 mm |
| 3 | position + DBH + height | as 2, plus a penalty when LiDAR height and field height differ by more than 5 m |

A missing value on either side never costs anything, and **the harvest record is used by none of the three**, so it stays free as an independent check. The LiDAR DBH of every candidate instance (not only the matched one) is needed; it is cached in `matching_comparison/instance_dbh.csv`. The chance control (stems shifted by 5 m) is run for each algorithm.

In [ ]:
COMPARISON = TABLES / 'matching_comparison'
INSTANCE_DBH_PATH = COMPARISON / 'instance_dbh.csv'

if RUN_MATCHING_COMPARISON:
    # Read back from the table, so this section sees exactly what the results notebook sees
    cmp_field = pd.read_csv(DETECTION_PATH, dtype={'tree_id': str})[
        ['tree_id', 'parzelle', 'species', 'utm_x', 'utm_y', 'dbh_mm', 'removed_2026', 'has_height',
         'height_vertex', 'height_massband', 'height_standing_sheet']].copy()
    cmp_field['dbh_mm'] = cmp_field['dbh_mm'].where(cmp_field['dbh_mm'].between(*DBH_PLAUSIBLE_MM))
    standing = cmp_field['height_standing_sheet'].where(cmp_field['height_standing_sheet'] <= STANDING_HEIGHT_DM_ABOVE,
                                                         cmp_field['height_standing_sheet'] / 10)
    cmp_field['height_felled'] = cmp_field[['height_vertex', 'height_massband']].mean(axis=1)
    cmp_field['height_standing'] = standing.where(cmp_field['height_felled'].isna())
    cmp_field['field_height_m'] = cmp_field['height_felled'].fillna(cmp_field['height_standing'])
    cmp_instances = pd.DataFrame(instances.drop(columns='geometry'))

    in_reach = cKDTree(cmp_instances[['x_base', 'y_base']].to_numpy()).query_ball_point(
        cmp_field[['utm_x', 'utm_y']].to_numpy(), r=MAX_MATCH_DISTANCE_M)
    candidates = cmp_instances.iloc[np.unique(np.concatenate([np.asarray(c, dtype=int) for c in in_reach]))]
    if FORCE_RECOMPUTE or not INSTANCE_DBH_PATH.exists():
        cloud = load_clouds()['before']
        t0 = time.time()
        breast_zone = cloud[(cloud['h'] > 0.8) & (cloud['h'] < 1.8)]
        fits = ftm.estimate_dbh(breast_zone, pd.DataFrame({'tree_id': candidates['instance'], 'x': candidates['x_base'], 'y': candidates['y_base']}))
        fits.rename(columns={'tree_id': 'instance'})[['instance', 'dbh_lidar_mm', 'dbh_quality', 'n_inliers', 'arc_coverage', 'rmse_mm']] \
            .to_csv(INSTANCE_DBH_PATH, index=False)
        print(f"DBH fits for {len(fits)} candidate instances ({time.time() - t0:.0f} s)")
        del breast_zone
    instance_dbh = pd.read_csv(INSTANCE_DBH_PATH)
    cmp_instances = cmp_instances.merge(instance_dbh[['instance', 'dbh_lidar_mm', 'dbh_quality']], on='instance', how='left')
    cmp_instances['dbh_lidar_all_mm'] = cmp_instances['dbh_lidar_mm']
    cmp_instances['dbh_lidar_mm'] = cmp_instances['dbh_lidar_mm'].where(cmp_instances['dbh_quality'] == 'ok')   # only good fits enter the matching

    cmp_stems = cmp_field[['tree_id', 'utm_x', 'utm_y', 'dbh_mm', 'field_height_m']]
    chance_rows = []
    for name, settings in ALGORITHMS.items():
        m = ftm.match_stems_with_attributes(cmp_stems, cmp_instances, max_distance_m=MAX_MATCH_DISTANCE_M, **settings)
        m = m.merge(cmp_field[['tree_id', 'parzelle', 'species', 'dbh_mm', 'removed_2026', 'height_felled', 'height_standing']], on='tree_id')
        m = m.merge(cmp_instances[['instance', 'height', 'removed_fraction', 'dbh_lidar_mm', 'dbh_lidar_all_mm', 'dbh_quality']]
                    .astype({'instance': 'Int64'}), on='instance', how='left')
        m['outcome'] = np.select([m['instance'].isna(), m['base_dist_m'] <= FOUND_DISTANCE_M],
                                 ['no match', f'found (base within {FOUND_DISTANCE_M} m)'],
                                 default=f'far match ({FOUND_DISTANCE_M}-{MAX_MATCH_DISTANCE_M:.0f} m)')
        m.to_csv(COMPARISON / f"matches_{name[0]}.csv", index=False)
        for dx, dy, label in [(0, 1, 'north'), (1, 0, 'east'), (0, -1, 'south'), (-1, 0, 'west')]:
            shifted = cmp_stems.assign(utm_x=cmp_stems['utm_x'] + dx * CONTROL_SHIFT_M, utm_y=cmp_stems['utm_y'] + dy * CONTROL_SHIFT_M)
            found = ftm.match_stems_with_attributes(shifted, cmp_instances, max_distance_m=MAX_MATCH_DISTANCE_M, **settings)['base_dist_m'] <= FOUND_DISTANCE_M
            chance_rows.append({'algorithm': name, 'shift': label, 'found_share': float(found.mean())})
        print(f"{name}: {m['outcome'].value_counts().to_dict()}")
    pd.DataFrame(chance_rows).to_csv(COMPARISON / 'chance_control.csv', index=False)
else:
    print("Skipped (RUN_MATCHING_COMPARISON = False)")

## 12. Stem-seeded refinement of single instances

Rebuilds the point set of a field tree whose SegmentAnyTree instance is wrong, starting from its stem (`modules/stem_seeded_refinement.py`). Typical case: a small tree under a tall canopy, where SegmentAnyTree found a piece of the thin stem but attached a neighbour's crown to it (tree `2_1027`: 12.3 m in the field, 27.0 m from its instance).

1. **Stem base**: the base of the instance matched to the field tree.
2. **Trace the lower stem** upward in 0.5 m layers, as far as it is hit by returns.
3. **Extend the axis** as a straight line; the thin upper stem and the inside of a shaded crown are usually not hit at all.
4. **Stems compete for the points**: every point goes to the tree whose stem it is connected to by the shortest path through the point cloud.
5. **Clusters connected to no stem** go to the tree if they lie within 2 m of its extended axis.
6. **Top**: the tree ends at a gap in the heights of its points. With a field height, the gap closest to it is taken.
7. **Closing the gap to the canopy above** (only the trees in `TOP_EXTENSION_TREES`, first code cell): a small tree under a tall neighbour often reaches into the neighbour's lowest branches, so its top goes to the neighbour and the first gap stops it too low. Starting from the result without the field height, the points are followed upward in 0.25 m layers inside a 1 m cylinder around the centre of the upper crown, as long as they are at least 3 times denser there than in the ring out to 2.5 m. The neighbour's crown base shows up as an evenly spread layer and ends the tree. The field height is not used. For these trees the saved point set and the figures are this result (`point_set = gap_closed`); the columns with and without the field height are kept as before.

Each tree is refined twice: with the field height choosing the top, and without it. The first is **not independent** of the field height. A refined tree is its own tier, `stem_seeded_refinement`; it is not a SegmentAnyTree result and enters no SegmentAnyTree statistic.

Field height = Maßband length, else Vertex, else the height measured on the standing tree.

`REFINE_TREES = 'auto'` (first code cell) refines every sample tree and every standing-height tree whose SegmentAnyTree height is more than `REFINE_OUTLIER_M` off. The sample trees whose SegmentAnyTree height is already right are the control: there the refinement must not make things worse. A list such as `['2_1027']` refines only those trees. The result is cached in `refined_trees.csv` as long as the set of trees (and of `TOP_EXTENSION_TREES`) stays the same.

Step 7 was set on tree `2_1027` alone (12.2 m against 12.2 m Maßband; without it 10.9 m) and is not yet tested on other trees.

In [ ]:
REFINED_DIR = OUT / 'refined_trees'
REFINED_PATH = TABLES / 'refined_trees.csv'
REFINE_SETTINGS = dict(search_above_m=4.0,          # how far above the field height the cloud is searched
                       orphan_radius_m=2.0,         # clusters connected to no stem: taken within this distance of the axis
                       max_crown_radius_m=3.5,      # limit for points connected to the tree's own stem
                       top_gap_m=0.5)               # height range without points that ends the tree
CLIP_RADIUS_M, COMPETITOR_RADIUS_M = 12.0, 7.0      # points loaded around each tree; neighbouring stems that compete for the points

detection = pd.read_csv(DETECTION_PATH, dtype={'tree_id': str, 'parzelle': str})
detection['height_standing'] = detection['height_standing_sheet'].where(detection['height_standing_sheet'] <= STANDING_HEIGHT_DM_ABOVE,
                                                                        detection['height_standing_sheet'] / 10)
# Field height: Maßband (the reference), else Vertex, else the standing height
detection['field_height'] = detection['height_massband'].fillna(detection['height_vertex']).fillna(detection['height_standing'])
detection['field_height_source'] = np.select(
    [detection['height_massband'].notna(), detection['height_vertex'].notna(), detection['height_standing'].notna()],
    ['Maßband', 'Vertex', 'standing height'], default='')
if isinstance(REFINE_TREES, str):
    assert REFINE_TREES == 'auto', "REFINE_TREES is 'auto' or a list of tree ids"
    standing_outlier = (~detection['has_height'] & detection['height_standing'].notna()
                        & ((detection['lidar_height'] - detection['height_standing']).abs() > REFINE_OUTLIER_M))
    refine_ids = detection.loc[(detection['has_height'] | standing_outlier) & detection['x_base'].notna(), 'tree_id'].tolist()
else:
    refine_ids = list(REFINE_TREES)
if REFINED_PATH.exists():
    _cached = pd.read_csv(REFINED_PATH, dtype={'tree_id': str})
    cached = (set(_cached['tree_id']) == set(refine_ids) and 'refined_height_gap_closed' in _cached.columns
              and set(_cached.loc[_cached['refined_height_gap_closed'].notna(), 'tree_id']) == set(TOP_EXTENSION_TREES) & set(refine_ids))
else:
    cached = False

if refine_ids and (FORCE_RECOMPUTE or not cached):
    todo = detection[detection['tree_id'].isin(refine_ids)].copy()
    missing = sorted(set(refine_ids) - set(todo['tree_id']))
    assert not missing, f"Not in the detection table: {missing}"
    assert todo['x_base'].notna().all(), f"No matched instance (no stem base) for: {todo.loc[todo['x_base'].isna(), 'tree_id'].tolist()}"
    print(f"Refining {len(todo)} trees")

    t0 = time.time()
    cloud = ssr.load_points_near(CLIP_BEFORE, todo[['x_base', 'y_base']].to_numpy(), CLIP_RADIUS_M, DTM_PATH, instance_dim=INSTANCE_DIM)
    after = ssr.load_points_near(CLIP_AFTER, todo[['x_base', 'y_base']].to_numpy(), CLIP_RADIUS_M, DTM_PATH)
    removed_near = ftm.flag_removed_points(cloud, after, (cloud['x'].min(), cloud['y'].min(), cloud['x'].max(), cloud['y'].max()), voxel_m=0.4)
    del after
    print(f"Loaded {len(cloud):,} points around the trees ({time.time() - t0:.0f} s)")

    rows = []
    for _, tree in todo.iterrows():
        t1 = time.time()
        seed = (tree['x_base'], tree['y_base'])
        others = detection[(detection['tree_id'] != tree['tree_id']) & detection['x_base'].notna()]
        others = others[np.hypot(others['x_base'] - seed[0], others['y_base'] - seed[1]) < COMPETITOR_RADIUS_M]
        competitors = pd.DataFrame({'x': others['x_base'], 'y': others['y_base'], 'dbh_mm': others['dbh_mm']})
        row = {'tree_id': tree['tree_id'], 'method': 'stem_seeded_refinement', 'detection': tree['detection'],
               'height_vertex': tree['height_vertex'], 'height_massband': tree['height_massband'], 'height_standing': tree['height_standing'],
               'field_height': tree['field_height'], 'field_height_source': tree['field_height_source'], 'dbh_mm': tree['dbh_mm'],
               'sat_instance': tree['instance'], 'sat_height': tree['lidar_height'], 'competing_stems': len(competitors)}
        try:
            guided = ssr.refine_tree_from_stem(cloud, seed, tree['dbh_mm'], competitors,
                                               expected_height_m=tree['field_height'], crown_base_m=tree['crown_base_m'], **REFINE_SETTINGS)
            free = ssr.refine_tree_from_stem(cloud, seed, tree['dbh_mm'], competitors, **REFINE_SETTINGS)
        except ValueError as error:          # no stem points at the base
            rows.append({**row, 'failed': str(error)})
            print(f"Tree {tree['tree_id']}: failed ({error})")
            continue
        m, f = guided['metrics'], free['metrics']
        # Point set that is saved and drawn: the gap-closed tree for TOP_EXTENSION_TREES, else the one with the field height
        if tree['tree_id'] in TOP_EXTENSION_TREES:
            gap = ssr.extend_top_into_canopy(cloud, free['mask'])
            shown, shown_axis, shown_connected = gap['mask'], free['axis'], free['connected']
            top = cloud.loc[shown, 'h'].idxmax()
            shown_metrics = {**f, 'height': gap['metrics']['height'], 'x_top': cloud.at[top, 'x'], 'y_top': cloud.at[top, 'y'],
                             'n_points': int(shown.sum()), 'top_chosen_by': 'gap_to_canopy_above'}
            gap_columns = {'refined_height_gap_closed': gap['metrics']['height'], 'n_top_added': gap['metrics']['n_added'],
                           'canopy_base_above_m': gap['metrics']['canopy_base_m'], 'point_set': 'gap_closed'}
        else:
            shown, shown_axis, shown_connected, shown_metrics = guided['mask'], guided['axis'], guided['connected'], m
            gap_columns = {'refined_height_gap_closed': np.nan, 'n_top_added': np.nan, 'canopy_base_above_m': np.nan,
                           'point_set': 'field_height'}
        laz_path = ssr.write_refined_tree(cloud, shown, REFINED_DIR / f"tree_{tree['tree_id']}.laz")
        rows.append({**row,
            'refined_height': m['height'], 'top_chosen_by': m['top_chosen_by'],
            'refined_height_without_field_height': f['height'], 'top_chosen_by_without': f['top_chosen_by'],
            'candidate_tops': '; '.join(f"{t:.2f}" for t in m['candidate_tops']),
            'x_top': m['x_top'], 'y_top': m['y_top'], 'x_base': m['x_base'], 'y_base': m['y_base'], 'lean_m': m['lean_m'],
            'traced_stem_top_m': m['traced_stem_top_m'], 'crown_radius_m': m['crown_radius_m'],
            'crown_base_field_m': tree['crown_base_m'], 'lowest_branch_lidar_m': m['lowest_branch_m'],
            'n_points': m['n_points'], 'n_connected': m['n_connected'], 'n_by_position': m['n_orphan'],
            'n_points_without': f['n_points'], 'n_by_position_without': f['n_orphan'], **gap_columns,
            'file': laz_path.name, 'failed': ''})
        print(f"Tree {tree['tree_id']}: field {tree['field_height']:.2f} m ({tree['field_height_source']}), SegmentAnyTree {tree['lidar_height']:.2f} m, "
              f"refined {m['height']:.2f} m with the field height, {f['height']:.2f} m without"
              + (f", {gap_columns['refined_height_gap_closed']:.2f} m with the gap to the canopy closed" if tree['tree_id'] in TOP_EXTENSION_TREES else '')
              + f" ({time.time() - t1:.0f} s)")

        # Two vertical slabs and a plan view of the refined tree, then the QA layout with the refined tree in purple
        fig = ssr.plot_refinement(cloud, shown, shown_axis, tree, shown_metrics,
                                  output_path=OUT / 'figures' / 'refined_trees' / f"tree_{tree['tree_id']}.png", connected=shown_connected)
        plt.close(fig)
        qa_tree = tree.copy()
        qa_tree['confidence'], qa_tree['method'] = tree['detection'], 'sat_instance'
        fig = ftm.plot_match_qa(qa_tree, cloud, removed_near, chm, chm_transform, ORTHO_BEFORE, ORTHO_AFTER,
                                output_path=OUT / 'figures' / 'refined_trees' / f"tree_{tree['tree_id']}_qa.png",
                                refined_points=cloud.loc[shown, ['x', 'y', 'h']])
        plt.close(fig)
    pd.DataFrame(rows).to_csv(REFINED_PATH, index=False)
    del cloud, removed_near
    print(f"Table: {REFINED_PATH}; point sets: {REFINED_DIR}; figures: {OUT / 'figures' / 'refined_trees'}")
elif refine_ids:
    print(f"Using cached {REFINED_PATH.name} ({len(refine_ids)} trees)")
else:
    print("No trees to refine (REFINE_TREES is empty)")

## 13. LiDAR instances without a field tree

359 of the 1906 instances with their base inside a Parzelle have no field tree, most of them 20–30 m tall. Each of them is either a **tree that is not in the inventory** (or whose field position is more than 3 m off), or a **piece of a neighbouring tree** that SegmentAnyTree split off. A real tree has a stem below its crown; a piece split off a neighbour usually has none. Evidence per instance:

- **stem at the base**: the stem search of section 6 (circle fits in eight slices from 1.5 to 8.5 m), started at the instance base; `stem_slices` ≥ `MIN_STEM_SLICES` counts as a stem, and `stem_is_own` says whether the stem points belong to this instance
- nearest field stem and whether that tree is matched to another instance
- nearest instance that has a field tree: distance of base and top, its height
- distance of the base to the Parzelle outline (a tree just outside the Parzelle may have its base drawn inside)

`reading` is a first rule on this evidence, to be checked on the figures:
1. own stem, no field stem within 1.5 m → *own stem, not in the inventory*
2. own stem, a field stem within 1.5 m → *own stem next to a field stem* (two instances on one stem, or a field position that belongs to the other one)
3. no stem, an instance with a field tree within 3 m → *no stem, next to a matched tree: probably split off*
4. otherwise → *no stem, no matched tree nearby*

`N_UNMATCHED_FIGURES` instances, drawn at random (seed 155), get a QA figure in `figures/qa_instances_without_field_tree/`: same layout as the field-tree figures, centred on the instance base (cyan cross = red cross here); white circles = field stems.

In [ ]:
UNMATCHED_PATH = TABLES / 'lidar_instances_without_field_tree.csv'
UNMATCHED_FIG = OUT / 'figures' / 'qa_instances_without_field_tree'

if FORCE_RECOMPUTE or not UNMATCHED_PATH.exists():
    lonely = in_plot[in_plot['tree_id'].isna()].drop(columns=['geometry', 'tree_id', 'detection']).copy()
    base_xy = lonely[['x_base', 'y_base']].to_numpy()

    # Nearest field stem, and whether that tree is matched to an instance
    field_xy = result[['utm_x', 'utm_y']].to_numpy()
    dist, idx = cKDTree(field_xy).query(base_xy)
    lonely['nearest_stem_m'] = dist
    lonely['nearest_tree_id'] = result['tree_id'].to_numpy()[idx]
    lonely['nearest_tree_detection'] = result['detection'].astype(str).to_numpy()[idx]
    lonely['nearest_tree_dbh_mm'] = result['dbh_mm'].to_numpy()[idx]
    lonely['field_stems_within_3m'] = cKDTree(field_xy).query_ball_point(base_xy, r=3.0, return_length=True)

    # Nearest instance that has a field tree
    with_tree = inst[inst['tree_id'].notna()]
    dist, idx = cKDTree(with_tree[['x_base', 'y_base']].to_numpy()).query(base_xy)
    lonely['nearest_matched_base_m'] = dist
    lonely['nearest_matched_height'] = with_tree['height'].to_numpy()[idx]
    lonely['nearest_matched_tree_id'] = with_tree['tree_id'].to_numpy()[idx]
    lonely['nearest_matched_top_m'] = cKDTree(with_tree[['x_top', 'y_top']].to_numpy()).query(lonely[['x_top', 'y_top']].to_numpy())[0]

    outline = plots.set_index('parzelle').geometry.boundary
    lonely['edge_dist_m'] = [outline[p].distance(Point(x, y)) for p, x, y in zip(lonely['parzelle'], lonely['x_base'], lonely['y_base'])]

    # Stem at the instance base
    t0 = time.time()
    near_base = ssr.load_points_near(CLIP_BEFORE, base_xy, 1.5, DTM_PATH, instance_dim=INSTANCE_DIM)
    near_base = near_base[(near_base['h'] > 1.0) & (near_base['h'] < 9.0)]
    stems_at_base = ftm.stem_profiles(near_base, pd.DataFrame({'tree_id': lonely['instance'], 'x': lonely['x_base'], 'y': lonely['y_base']}))
    print(f"Stem search at {len(lonely)} instance bases ({time.time() - t0:.0f} s)")
    lonely = lonely.merge(stems_at_base[['tree_id', 'stem_slices', 'stem_offset_m', 'stem_instance', 'stem_instance_share']]
                          .rename(columns={'tree_id': 'instance'}), on='instance', how='left')
    lonely['stem_is_own'] = lonely['stem_instance'] == lonely['instance']
    has_stem = lonely['stem_slices'] >= MIN_STEM_SLICES
    lonely['reading'] = np.select(
        [has_stem & (lonely['nearest_stem_m'] > HIGH_CONFIDENCE_DISTANCE_M), has_stem,
         lonely['nearest_matched_base_m'] <= MAX_MATCH_DISTANCE_M],
        ['own stem, not in the inventory', 'own stem next to a field stem', 'no stem, next to a matched tree: probably split off'],
        default='no stem, no matched tree nearby')
    lonely['edge'] = lonely['edge_dist_m'] < EDGE_M

    rng = np.random.default_rng(155)
    lonely['figure'] = False
    lonely.loc[rng.choice(lonely.index, size=min(N_UNMATCHED_FIGURES, len(lonely)), replace=False), 'figure'] = True
    lonely.to_csv(UNMATCHED_PATH, index=False)
else:
    print(f"Using cached {UNMATCHED_PATH.name}")

lonely = pd.read_csv(UNMATCHED_PATH)
print(f"{len(lonely)} instances without a field tree:", lonely['reading'].value_counts().to_dict())

drawn = lonely[lonely['figure']]
todo = drawn[[FORCE_RECOMPUTE or not (UNMATCHED_FIG / f"instance_{int(i)}.png").exists() for i in drawn['instance']]]
if len(todo):
    t0 = time.time()
    centres = todo[['x_base', 'y_base']].to_numpy()
    cloud = ssr.load_points_near(CLIP_BEFORE, centres, 10.5, DTM_PATH, instance_dim=INSTANCE_DIM)
    after = ssr.load_points_near(CLIP_AFTER, centres, 10.5, DTM_PATH)
    removed_near = ftm.flag_removed_points(cloud, after, (cloud['x'].min(), cloud['y'].min(), cloud['x'].max(), cloud['y'].max()), voxel_m=0.4)
    del after
    for _, row in todo.iterrows():
        pseudo = pd.Series({'tree_id': f"instance {int(row['instance'])}", 'utm_x': row['x_base'], 'utm_y': row['y_base'],
                            'x_base': row['x_base'], 'y_base': row['y_base'], 'x_top': row['x_top'], 'y_top': row['y_top'],
                            'instance': int(row['instance']), 'lidar_height': row['height'], 'confidence': row['reading'],
                            'method': 'no field tree', 'base_dist_m': np.nan, 'removed_2026': np.nan, 'removed_fraction': row['removed_fraction']})
        close = np.hypot(result['utm_x'] - row['x_base'], result['utm_y'] - row['y_base']) < 10.5
        fig = ftm.plot_match_qa(pseudo, cloud, removed_near, chm, chm_transform, ORTHO_BEFORE, ORTHO_AFTER,
                                output_path=UNMATCHED_FIG / f"instance_{int(row['instance'])}.png",
                                other_stems=pd.DataFrame({'x': result.loc[close, 'utm_x'], 'y': result.loc[close, 'utm_y']}))
        plt.close(fig)
    del cloud, removed_near
    print(f"Drew {len(todo)} figures in {UNMATCHED_FIG} ({time.time() - t0:.0f} s)")

## 14. Baseline: local maxima on the LiDAR CHM

Does SegmentAnyTree find more trees than the simplest established method? The baseline finds tree tops as local maxima of the LiDAR canopy height model (0.25 m, gaps filled, smoothed with σ = 0.25 m; `modules/chm_baseline.py`), with several windows (`BASELINE_WINDOWS`): fixed diameters from 1.5 to 4 m, and one that grows with the height. A small window finds more tops (also false ones), a large one fewer.

A CHM method has no stem base, so the comparison has to be made on the **tops**: every method's tops, and the tops of the SegmentAnyTree instances as well, are assigned to the field stems with the same one-to-one assignment as in section 4 (largest distance `TOP_MATCH_MAX_M`, no harvest penalty). A field tree counts as found if its top lies within `TOP_FOUND_DISTANCE_M` of the stem. The stems shifted by 5 m give the chance level, as in section 4. Statistics are in section 3.6 of the results notebook.

In [ ]:
BASELINE_TOPS_PATH = TABLES / 'chm_baseline_tops.csv'
BASELINE_MATCH_PATH = TABLES / 'chm_baseline_matches.csv'

if FORCE_RECOMPUTE or not (BASELINE_TOPS_PATH.exists() and BASELINE_MATCH_PATH.exists()):
    t0 = time.time()
    all_tops = [chm_baseline.local_maxima_tops(chm, chm_transform, window, min_height_m=BASELINE_MIN_HEIGHT_M,
                                               smoothing_sigma_m=BASELINE_SMOOTHING_M).assign(method=name)
                for name, window in BASELINE_WINDOWS.items()]
    all_tops.append(instances[['instance', 'x_top', 'y_top', 'height']].rename(columns={'instance': 'top_id'})
                    .assign(method='SegmentAnyTree instance tops'))
    all_tops = pd.concat(all_tops, ignore_index=True)
    located = gpd.sjoin(gpd.GeoDataFrame(all_tops, geometry=gpd.points_from_xy(all_tops['x_top'], all_tops['y_top']), crs=CRS),
                        plots[['parzelle', 'geometry']], predicate='within', how='left')
    all_tops['parzelle'] = located.loc[~located.index.duplicated(), 'parzelle'].to_numpy()
    all_tops.to_csv(BASELINE_TOPS_PATH, index=False)

    rows = []
    for name, tops in all_tops.groupby('method', sort=False):
        candidates = pd.DataFrame({'instance': tops['top_id'], 'x_base': tops['x_top'], 'y_base': tops['y_top'],
                                   'x_top': tops['x_top'], 'y_top': tops['y_top'], 'height': tops['height'],
                                   'removed_fraction': 0.0, 'lean_m': 0.0, 'crown_base_m': np.nan})
        for dx, dy, shift in [(0, 0, 'none'), (0, 1, 'north'), (1, 0, 'east'), (0, -1, 'south'), (-1, 0, 'west')]:
            shifted = stems.assign(utm_x=stems['utm_x'] + dx * CONTROL_SHIFT_M, utm_y=stems['utm_y'] + dy * CONTROL_SHIFT_M)
            reach = cKDTree(shifted[['utm_x', 'utm_y']].to_numpy()).query(candidates[['x_base', 'y_base']].to_numpy(),
                                                                         distance_upper_bound=TOP_MATCH_MAX_M)[0]
            assigned = ftm.match_field_trees_to_instances(
                shifted, candidates[np.isfinite(reach)], max_distance_m=TOP_MATCH_MAX_M, removal_penalty_m=0.0,
                high_confidence_distance_m=TOP_FOUND_DISTANCE_M)
            rows.append(assigned[['tree_id', 'instance', 'base_dist_m', 'lidar_height']]
                        .rename(columns={'instance': 'top_id', 'base_dist_m': 'top_dist_m', 'lidar_height': 'top_height'})
                        .assign(method=name, shift=shift))
        print(f"{name}: {len(tops)} tops ({time.time() - t0:.0f} s)")
    pd.concat(rows, ignore_index=True).to_csv(BASELINE_MATCH_PATH, index=False)
else:
    print(f"Using cached {BASELINE_TOPS_PATH.name} and {BASELINE_MATCH_PATH.name}")

baseline_tops = pd.read_csv(BASELINE_TOPS_PATH)
baseline_matches = pd.read_csv(BASELINE_MATCH_PATH, dtype={'tree_id': str})
found = baseline_matches[baseline_matches['shift'] == 'none'].groupby('method', sort=False)['top_dist_m'].apply(lambda d: (d <= TOP_FOUND_DISTANCE_M).mean())
print(pd.DataFrame({'tops': baseline_tops.groupby('method', sort=False).size(), 'field trees found': found.round(3)}).to_string())

## 15. The 18 Mar flight (double grid): harvest state and DBH (slow the first time, then cached)

The 18 Mar flight was flown in a double grid (about twice the points of 24 Mar, three times as many at breast height), after the harvest had started. This section prepares two comparisons for `02_results_155.ipynb`:
- **Harvest date**: for every 25 Feb instance the share of its points (above 2 m) that are gone on 18 Mar, with the same voxel test as `removed_fraction` against 24 Mar (section 2). The 24 Mar value is written again so both come from one place.
- **DBH in all three flights**: the points 0.8–1.8 m above the ground of each flight (all on the 24 Mar DTM; the lowest returns of both March flights lie at −0.05 m on it), and the circle fit of section 8 at the **same position for every flight**: the 25 Feb stem base of every detected tree. Differences in DBH are then due to the points, not to the segmentation or the matching.

Writes `flight_20260318/instance_removed_by_date.csv`, `breast_slice_*.npz`, `dbh_20260225.csv`, `dbh_20260318.csv`, `dbh_20260324.csv`. The 18 Mar file is read in full (760 M points, about 7 minutes) and only the AOI is kept.

In [ ]:
import laspy

FLIGHT_18 = OUT / 'flight_20260318'
REMOVED_BY_DATE_PATH = FLIGHT_18 / 'instance_removed_by_date.csv'
SLICE_PATHS = {d: FLIGHT_18 / f'breast_slice_{d}.npz' for d in ['20260225', '20260318', '20260324']}
CACHE_INFO_PATH = FLIGHT_18 / 'cache_info.json'      # grid the cache was made on; a cache from another grid is recomputed
CACHE_INFO = {'aoi': [round(float(v), 3) for v in aoi], 'voxel_m': REMOVED_VOXEL_M, 'dtm': DTM_PATH.name}


def stream_aoi(path, instance_dim=None):
    """x, y, z (and the tree instance) and height above ground of the points of a full flight inside the AOI, read in chunks."""
    parts = []
    with laspy.open(path) as reader:
        dims = {d.lower(): d for d in reader.header.point_format.dimension_names}
        for chunk in reader.chunk_iterator(25_000_000):
            x, y = np.asarray(chunk.x), np.asarray(chunk.y)
            keep = (x >= aoi[0]) & (x < aoi[2]) & (y >= aoi[1]) & (y < aoi[3])
            part = {'x': x[keep], 'y': y[keep], 'z': np.asarray(chunk.z, dtype=np.float32)[keep]}
            if instance_dim:
                part['instance'] = np.asarray(chunk[dims[instance_dim.lower()]], dtype=np.int32)[keep]
            parts.append(pd.DataFrame(part))
    cloud = pd.concat(parts, ignore_index=True)
    ftm.add_height_above_ground(cloud, DTM_PATH)
    return cloud


if BEFORE_FLIGHT != '20260225':
    print("Skipped: this section compares the 18 Mar flight with the 25 Feb instances")
elif (FORCE_RECOMPUTE or not (REMOVED_BY_DATE_PATH.exists() and all(p.exists() for p in SLICE_PATHS.values()))
      or not CACHE_INFO_PATH.exists() or json.loads(CACHE_INFO_PATH.read_text()) != CACHE_INFO):
    FLIGHT_18.mkdir(exist_ok=True)
    clouds = load_clouds(with_after=True)
    before = clouds['before']
    upper = (before['h'].to_numpy() > 2.0) & (before['instance'].to_numpy() > 0)      # as removed_fraction in section 2
    t0 = time.time()
    mar18 = stream_aoi(BEFORE_FLIGHTS['20260318'][0])
    print(f"18 Mar: {len(mar18):,} points in the AOI ({time.time() - t0:.0f} s)")
    removed_18 = ftm.flag_removed_points(before, mar18, aoi, voxel_m=REMOVED_VOXEL_M)
    by_date = pd.DataFrame({
        'removed_20260318': pd.Series(removed_18[upper]).groupby(before['instance'].to_numpy()[upper]).mean(),
        'removed_20260324': pd.Series(clouds['removed'][upper]).groupby(before['instance'].to_numpy()[upper]).mean()}).rename_axis('instance')
    by_date.to_csv(REMOVED_BY_DATE_PATH)
    for d, cloud in [('20260225', before), ('20260318', mar18), ('20260324', clouds['after'])]:
        s = cloud.loc[(cloud['h'] > 0.8) & (cloud['h'] < 1.8), ['x', 'y', 'h']]
        np.savez(SLICE_PATHS[d], **{c: s[c].to_numpy() for c in s.columns})
        print(f"{d}: {len(s):,} points at breast height")
    CACHE_INFO_PATH.write_text(json.dumps(CACHE_INFO))
    for d in SLICE_PATHS:
        (FLIGHT_18 / f'dbh_{d}.csv').unlink(missing_ok=True)      # the DBH below is redone on the new slices
    del before, mar18, removed_18, cloud, s
else:
    print(f"Using cached {REMOVED_BY_DATE_PATH.name} and breast-height slices")

if BEFORE_FLIGHT == '20260225':
    # DBH at the same positions (25 Feb stem base) in all three flights, for every detected tree with an instance
    detected = result[(result['detection'] == 'detected') & result['x_base'].notna()]
    centres = pd.DataFrame({'tree_id': detected['tree_id'], 'x': detected['x_base'], 'y': detected['y_base']})
    for d, path in SLICE_PATHS.items():
        dbh_path = FLIGHT_18 / f'dbh_{d}.csv'
        if FORCE_RECOMPUTE or not dbh_path.exists():
            s = pd.DataFrame(dict(np.load(path)))
            ftm.estimate_dbh(s, centres, breast_height_m=DBH_BREAST_HEIGHT_M, slice_half_height_m=DBH_SLICE_HALF_M,
                             search_radius_m=DBH_SEARCH_RADIUS_M).to_csv(dbh_path, index=False)
        print(f"DBH {d}: {pd.read_csv(dbh_path)['dbh_quality'].value_counts().to_dict()}")

### 15b. SegmentAnyTree in the two March flights (slow the first time, then cached)

Both March flights were segmented by SegmentAnyTree on their own (24 Mar: `PredInstance`, 18 Mar: `PredInstance_SAT`). Their instances are summarized as in section 2 and the standing field trees are matched to each flight with the same one-to-one base matching as section 4 (no harvest penalty: everything counted is standing), plus the chance control with stems shifted by 5 m.

*Standing* comes from the 25 Feb instance of the tree (more than half of its points gone on that date = felled; section 15), or from the removal mark if the tree has no 25 Feb instance; it does not use the March segmentations. On 18 Mar the trees felled after 18 Mar (Parzellen 1–11) still stand: they take part in the 18 Mar matching so their instances are not given to a neighbour, but the results notebook counts only trees standing on 24 Mar.

Writes `flight_20260318/instances_20260318.csv`, `instances_20260324.csv`, `match_by_flight.csv`, `chance_control_by_flight.csv`.

In [ ]:
MARCH_FLIGHTS = {'20260318': BEFORE_FLIGHTS['20260318'][1], '20260324': 'PredInstance'}     # flight -> instance field
MARCH_INSTANCE_PATHS = {d: FLIGHT_18 / f'instances_{d}.csv' for d in MARCH_FLIGHTS}
MARCH_MATCH_PATH = FLIGHT_18 / 'match_by_flight.csv'
MARCH_CONTROL_PATH = FLIGHT_18 / 'chance_control_by_flight.csv'

if BEFORE_FLIGHT == '20260225':
    if FORCE_RECOMPUTE or not all(p.exists() for p in MARCH_INSTANCE_PATHS.values()):
        for d, dim in MARCH_FLIGHTS.items():
            t0 = time.time()
            if d == '20260324':
                cloud = load_clouds(with_after=True)['after']
            else:
                cloud = stream_aoi(BEFORE_FLIGHTS['20260318'][0], instance_dim=dim)
            march_instances = ftm.summarize_instances(cloud, None, crs=CRS)
            pd.DataFrame(march_instances.drop(columns='geometry')).to_csv(MARCH_INSTANCE_PATHS[d], index=False)
            print(f"{d}: {len(march_instances)} instances ({time.time() - t0:.0f} s)")
            del cloud, march_instances
    march_instances = {d: pd.read_csv(p).assign(removed_fraction=0.0) for d, p in MARCH_INSTANCE_PATHS.items()}

    # Which field trees stand on which date: from the 25 Feb instance (section 15), else from the removal mark
    by_date = pd.read_csv(REMOVED_BY_DATE_PATH, index_col='instance')
    gone_18 = result['instance'].map(by_date['removed_20260318']) > REMOVED_THRESHOLD
    gone_24 = result['instance'].map(by_date['removed_20260324']) > REMOVED_THRESHOLD
    no_feb_instance = result['instance'].isna()
    standing_24 = np.where(no_feb_instance, ~result['removed_2026'], ~gone_24)
    standing_18 = np.where(no_feb_instance, ~result['removed_2026'], ~gone_18)
    stand = result[['tree_id', 'parzelle', 'utm_x', 'utm_y']].assign(standing_18=standing_18.astype(bool), standing_24=standing_24.astype(bool),
                                                                    removed_2026=False)

    rows, control_rows = [], []
    for d in MARCH_FLIGHTS:
        stems = stand[stand['standing_24'] | (stand['standing_18'] if d == '20260318' else False)]   # 18 Mar: also the trees felled later
        for dx, dy, label in [(0, 0, 'none'), (0, 1, 'north'), (1, 0, 'east'), (0, -1, 'south'), (-1, 0, 'west')]:
            shifted = stems.assign(utm_x=stems['utm_x'] + dx * CONTROL_SHIFT_M, utm_y=stems['utm_y'] + dy * CONTROL_SHIFT_M)
            m = ftm.match_field_trees_to_instances(shifted, march_instances[d], max_distance_m=MAX_MATCH_DISTANCE_M,
                                                   removed_threshold=REMOVED_THRESHOLD, removal_penalty_m=0.0,
                                                   high_confidence_distance_m=HIGH_CONFIDENCE_DISTANCE_M)
            m = m[['tree_id', 'instance', 'base_dist_m', 'lidar_height', 'x_base', 'y_base']].assign(flight=d, shift=label)
            (rows if label == 'none' else control_rows).append(m)
    march_match = pd.concat(rows, ignore_index=True).merge(stand[['tree_id', 'standing_18', 'standing_24']], on='tree_id')
    march_match.to_csv(MARCH_MATCH_PATH, index=False)
    pd.concat(control_rows, ignore_index=True).to_csv(MARCH_CONTROL_PATH, index=False)
    found = march_match[march_match['standing_24']].groupby('flight')['base_dist_m'].apply(lambda s: (s <= HIGH_CONFIDENCE_DISTANCE_M).mean())
    print(f"Field trees standing on 24 Mar: {int(stand['standing_24'].sum())}; base within {HIGH_CONFIDENCE_DISTANCE_M} m: "
          + ', '.join(f"{d} {v:.1%}" for d, v in found.items()))

In [ ]:
_clouds.clear()      # free the point clouds
print("Pipeline finished. Continue with 02_results_155.ipynb.")